# 1 Why you should not run the encoder twice

This course starts one level up from where it used to. Not "here is a tensor, here is a
loss, here is MNIST", but: **download a representation someone else already paid to
compute, and spend your time on the question that actually matters.**

Everything you need from the old first lesson — shapes, a loss, an optimizer, a training
loop — still gets taught. It gets taught in the notebook where it is needed, against data
that is already interesting.

The pipeline has two halves and they have wildly different price tags:

| | what it is | cost | how often |
|---|---|---|---|
| **encode** | a frozen pretrained model turns raw data into vectors | expensive, wants a GPU | **once** |
| **head** | a small trainable network on top of those vectors | seconds, on any laptop | hundreds of times |

Running the encoder inside your training loop means recomputing identical vectors, every
epoch, to answer questions that have nothing to do with the encoder. Split the two and the
expensive half becomes a file you download.

That file is a `VectorCache`, and this notebook is about reading one.

In [ ]:
import json
import sys
import time

import datasets
import torch
from loguru import logger
from vectormesh import VectorCache

from notebooktester import param

# the library narrates every cache load; useful once, noise in a loop
logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

torch.manual_seed(42)

## 1.1 The hook: no training at all

Before any model code, a demonstration. Oxford Flowers-102 — 102 species of flower, a
genuinely fine-grained task. We load a published cache of DINOv2 vectors and classify the
test set with **1-nearest-neighbour**: for each test vector, find the closest training
vector by cosine similarity and copy its label.

No loss. No optimizer. No parameters. Just distances.

In [ ]:
tr = VectorCache.from_hub("pttrn-io/oxford-flowers-dinov2-small", split="train")
te = VectorCache.from_hub("pttrn-io/oxford-flowers-dinov2-small", split="test")

# cache["col"] is a lazy Column; the [:] materialises it into one tensor
Xtr, ytr = tr["embed"][:], tr["label"][:]
Xte, yte = te["embed"][:], te["label"][:]

start = time.time()
A = torch.nn.functional.normalize(Xtr, dim=1)     # cosine similarity = dot product
B = torch.nn.functional.normalize(Xte, dim=1)     # once both sides are unit length
neighbour = (B @ A.T).argmax(dim=1)               # (1000, 4000) -> index of the closest
accuracy = (ytr[neighbour] == yte).float().mean()

print(f"train {tuple(Xtr.shape)}   test {tuple(Xte.shape)}")
print(f"1-nearest-neighbour on 102 classes: {accuracy:.4f}")
print(f"chance level: {1 / 102:.4f}")
print(f"took {1000 * (time.time() - start):.0f} ms, trained 0 parameters")

99.1%, with nothing that could be called a model. In §1.5 we train an actual head on these
same vectors, and it lands on **the same number**. Give it a hidden layer and it reaches
99.3% — the entire trainable part of the pipeline is worth two tenths of a point.

Sit with that for a second, because it is the thesis of the whole course:

> **The representation is doing the work, and someone else already paid for it.**

A number like that on 102 classes deserves suspicion, so it was checked. Train and test
come from different source splits, the test set is balanced at 8–11 images per class, and
near-duplicates are too rare to carry the result. The decisive check is the other encoder:
`resnet-18` gets 90% through the *identical* rows and the *identical* head. A leak would
lift every encoder, not one.

Flowers is *saturated* — which makes it a perfect demo and a useless exercise. You cannot
learn anything from tuning a task that is already solved. The rest of this lesson works on
EuroSAT (satellite imagery, 94% — impressive, not solved) and DTD (textures, 78% — real
headroom).

## 1.2 What a cache actually is

`from_hub` downloaded a folder. Two things are in it: an arrow dataset with the vectors,
and a `metadata.json`.

**`metadata.json` is the public API of a dataset.** It is how you find out the shapes you
are about to receive, from a cache you did not build, before you write a line of model
code. This habit gets repeated in every lesson of this course.

In [ ]:
eurosat = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="train")
eurosat_test = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="test")

print(json.dumps(eurosat.metadata, indent=2))

Read it field by field, because each one answers a question you would otherwise answer by
running something and looking at a traceback:

| field | tells you |
|---|---|
| `model_tag` | which encoder produced these vectors — the thing that matters most |
| `hidden_size` | `384`. This is the **input width of your head**. |
| `tensordtype` | `1` — one item is a rank-1 `(D)` tensor, so batching is a plain `stack` |
| `num_observations` | how many rows you have to train on |
| `features` | which columns exist at all |

`tensordtype` is the one people skip. It says how many axes *one item* has. Here it is 1,
so an item is `(384,)` and a batch is `(B, 384)`. For chunked text (lesson 5) it is 2: an
item is `(C, D)`, a batch is `(B, C, D)`, and a plain head will not do — you need an
aggregator first. The metadata tells you which world you are in before you find out the
hard way.

The other two columns are not vectors:

- **`label`** — a `ClassLabel`, so it carries the class *names*, not just integers.
- **`source_idx`** — the row number in the original dataset. The cache was shuffled, so
  this is **not** the row number in the cache. It is how you get the picture back
  (exercise 2).

In [ ]:
print("columns      :", eurosat.column_names)
print("vector cols  :", eurosat.vector_columns)
print("rows         :", len(eurosat), "train,", len(eurosat_test), "test")
print()

labels = eurosat.features["label"]
print(f"{labels.num_classes} classes:")
for i, name in enumerate(labels.names):
    print(f"  {i}  {name}")

Ten classes of satellite imagery. Note what is *not* in this cache: pixels. The whole
train split is a few megabytes of float32, which is why you could download it in the time
it takes to read this paragraph — and why a classroom of thirty people can each have a
copy.

## 1.3 Tensors, as they actually turn up

This is not a tour of the torch API. It is the four facts you need to hold a cache row.

In [ ]:
row = eurosat[0]
for name, value in row.items():
    print(f"{name:12s} shape {tuple(value.shape)!s:8s} dtype {value.dtype}")

x = row["embed"]
print("\n==== embeded vector x ====")
print("x.shape       :", x.shape, 38*" ", "-- 384 numbers describing one satellite image")
print("x.dtype       :", x.dtype, 42*" ", "-- float32, the default for everything in this course")
print("x.device      :", x.device,52*" ", "-- where it lives; .to('mps') / .to('cuda') moves it")
print("index (x[:5]) :", x[:5], 2*" ", "-- and it indexes like anything else:")

### The rank ladder, image version

For a cached image pipeline the ladder is three rungs and you climb all of them in this
lesson:

| rank | shape | name | where it comes from |
|---|---|---|---|
| 1D | `(D)` | **vector** | one row of the cache |
| 2D | `(B, D)` | **batch of vectors** | after `Collate(padder=torch.stack)` |
| 2D | `(B, K)` | **logits** | after the head |

That is the whole trip: `(384,) → (B, 384) → (B, 10)`. Text gets one extra rung because a
long document is chunked into `(C, D)` before it is batched — that is lesson 5's problem,
and deliberately not yours yet.

The full treatment, including *why* `(C, D)` and `(B, D)` are indistinguishable to torch,
is in [tensor-ladder.md](tensor-ladder.md). Read it once; every later lesson refers back to
it instead of re-explaining.

### Shape is not meaning

Here is the trap that page exists for. `nn.Linear` acts on the **last** axis and silently
accepts any leading shape:

In [ ]:
head = torch.nn.Linear(384, 10)

print("(B, D)     ", tuple(head(torch.rand(32, 384)).shape), "  <- one prediction per item, what you meant")
print("(B, C, D)  ", tuple(head(torch.rand(32, 7, 384)).shape), "<- one prediction per *chunk*, no error raised")

The second call is formally fine and semantically a bug. It will train. The loss will go
down. Nothing will warn you.

The fix is to write the meaning into the code, where it can be checked:

In [ ]:
from beartype import beartype
from jaxtyping import Float, jaxtyped
from torch import Tensor


@jaxtyped(typechecker=beartype)
def classify(x: Float[Tensor, "batch dim"]) -> Float[Tensor, "batch classes"]:
    return head(x)


print("correct rung:", tuple(classify(torch.rand(32, 384)).shape))

try:
    classify(torch.rand(32, 7, 384))
except Exception as e:
    print(f"\nwrong rung raises {type(e).__name__}:")
    for line in str(e).splitlines():
        if "Actual" in line or "Expected" in line:
            print("   ", line.strip())

The names inside `Float[Tensor, "batch dim"]` are yours to choose — they are a claim about
what the axes *mean*, enforced at runtime. A shape error from a contract like this is not
noise; it is the only mechanism that turns a semantic mistake into a formal one, which is
the only kind a computer can catch for you.

## 1.4 The economics, with a stopwatch

You have been told encoding is expensive. Measure it, on your own machine, so the number
is yours and not the instructor's.

We encode **200 EuroSAT images** — deliberately small, so this finishes on a CPU-only VM.

In [ ]:
from datasets import load_dataset
from vectormesh import ImageVectorizer
from vectormesh.data.vectorizers import detect_device

N_IMAGES = param(200, test=16)

raw = load_dataset("tanganke/eurosat", split="test").select(range(N_IMAGES))
images = [row["image"] for row in raw]
print(f"{len(images)} images, first one is {images[0].size} pixels, mode {images[0].mode}")
print(f"device: {detect_device()}")

In [ ]:
vectorizer = ImageVectorizer(model_name="facebook/dinov2-small")

vectorizer(images[:4], batchsize=4)          # warm-up: the first call pays for setup

start = time.time()
out = vectorizer(images, batchsize=32)
elapsed = time.time() - start

rate = len(images) / elapsed
print(f"encoded {len(images)} images in {elapsed:.1f}s  ->  {rate:.0f} img/s")
print(f"one image becomes {tuple(out['embed'][0].shape)}")

That `LOAD REPORT` above, with its `UNEXPECTED` keys, is not an error. It is the library
telling you that the checkpoint contained a classification head which is being thrown
away — which is exactly what we want. We are taking the *body* of the network, the part
that turns pixels into a description, and discarding its opinion about ImageNet classes.

Now time the other half: training a head over the **entire** 6000-row cache.

In [ ]:
from torch import nn

Xtr = eurosat["embed"][:]
ytr = eurosat["label"][:]

model = nn.Linear(384, 10)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

start = time.time()
for _ in range(param(10, test=1)):
    for i in range(0, len(Xtr), 256):
        opt.zero_grad()
        loss_fn(model(Xtr[i:i + 256]), ytr[i:i + 256]).backward()
        opt.step()
head_secs = time.time() - start

Xte, yte = eurosat_test["embed"][:], eurosat_test["label"][:]
with torch.no_grad():
    acc = (model(Xte).argmax(1) == yte).float().mean()

print(f"trained a head over all {len(Xtr)} rows in {head_secs:.2f}s, test accuracy {acc:.4f}")

In [ ]:
CACHE_ROWS = 8000                    # eurosat: 6000 train + 2000 test
EXPERIMENTS_THIS_WEEK = 50

encode_once = CACHE_ROWS / rate
in_loop = EXPERIMENTS_THIS_WEEK * (encode_once + head_secs)
cached = encode_once + EXPERIMENTS_THIS_WEEK * head_secs

print(f"your encoder rate       : {rate:>9.0f} img/s")
print(f"encoding eurosat once   : {encode_once:>9.1f} s")
print(f"training one head       : {head_secs:>9.2f} s")
print()
print(f"{EXPERIMENTS_THIS_WEEK} experiments, encoder in the loop : {in_loop / 60:>7.1f} min")
print(f"{EXPERIMENTS_THIS_WEEK} experiments, encoded once       : {cached / 60:>7.1f} min")
print(f"speedup                            : {in_loop / cached:>7.0f}x")

**Compare your `img/s` with the person next to you.** The spread across the room is the
lesson. On a fast Apple-silicon machine this is around ~80 img/s on CPU and ~250 on MPS or GPU; on
an older laptop or a basic cloud VM, 10–30 img/s is normal.

Multiply your own number by the three encoders and two datasets this lesson uses, and you
get somewhere between ninety seconds and most of an hour — repeated by every student, to
reproduce a byte-identical file that already exists.

So: **no notebook in this lesson builds a full cache.** Every one downloads a published
one. That is not a concession to weak hardware, it is the architecture being taught. If
you *do* have the hardware and want to encode something nobody has published yet,
`make build-lesson1` is there — but building a cache is lesson 2's subject.

The asymmetry is real and it exists in industry too. Encoding is the part you buy. Heading
is the part you think about, and that is the part this course is about.

## 1.5 What actually moves the number

Same rows, same head, same everything — three different encoders. This is the comparison
to remember when notebook 04 searches over architectures and finds nothing.

In [ ]:
def two_line_head(train: VectorCache, test: VectorCache, epochs: int = param(10, test=1)) -> tuple[float, int]:
    # one Linear over cached vectors -- deliberately the dumbest possible head
    Xtr, ytr = train["embed"][:], train["label"][:]
    Xte, yte = test["embed"][:], test["label"][:]

    torch.manual_seed(0)
    model = nn.Linear(Xtr.shape[1], int(train.features["label"].num_classes))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 256):
            idx = perm[i:i + 256]
            opt.zero_grad()
            loss_fn(model(Xtr[idx]), ytr[idx]).backward()
            opt.step()

    with torch.no_grad():
        return float((model(Xte).argmax(1) == yte).float().mean()), Xtr.shape[1]


ENCODERS = ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"]

print(f"{'encoder':<24} {'dim':>5} {'flowers':>9} {'eurosat':>9}")
print("-" * 50)
for enc in ENCODERS:
    f_acc, dim = two_line_head(
        VectorCache.from_hub(f"pttrn-io/oxford-flowers-{enc}", split="train"),
        VectorCache.from_hub(f"pttrn-io/oxford-flowers-{enc}", split="test"),
    )
    e_acc, _ = two_line_head(
        VectorCache.from_hub(f"pttrn-io/eurosat-{enc}", split="train"),
        VectorCache.from_hub(f"pttrn-io/eurosat-{enc}", split="test"),
    )
    print(f"{enc:<24} {dim:>5} {f_acc:>9.4f} {e_acc:>9.4f}")

Three things in that table, and the last two are easy to miss.

**The encoder choice moves flowers by fifteen points.** Same rows, same head, same seed —
the only difference is which frozen model produced the numbers. That is an axis with real
leverage on it, and you did not train anything to get it.

**`mobilenet_v2` has 1280 dimensions against DINOv2's 384, and loses by eleven.** More
numbers is not more information. Width is not quality.

**The spread is not the same on both datasets.** Flowers separates the encoders cleanly;
EuroSAT compresses them into about three points. Satellite imagery apparently sits close
enough to what all three encoders already learned that the choice barely matters. Worth
knowing *before* you design an experiment: an axis that dominates on one dataset can be
flat on another, and a search along a flat axis tells you nothing. Notebook 04 runs its
search on DTD for exactly this reason.

And compare the flowers column against §1.1. A trained linear head gets `0.9910` on
flowers — the same number 1-nearest-neighbour got with no training at all. The head is not
adding information. It is reading information the encoder already put there.

## What you now know

- A cache is vectors plus a `metadata.json`, and the metadata is the API. Read it first.
- `hidden_size` is your head's input width; `tensordtype` tells you the rank of one item.
- The rank ladder for images is `(D) → (B, D) → (B, K)`, and shape does not imply meaning
  — write contracts when the meaning matters.
- Encoding costs real time and happens once; heading is free and happens constantly.
- The representation, not the architecture on top of it, is where the leverage is.

## Exercises

**E1 — read a cache you did not build.** Pick `pttrn-io/dtd-mobilenet_v2_1.0_224`. From
its metadata alone, state (a) the input width of a head for it, (b) the shape one batch of
32 will have, (c) how many classes it has, and (d) which encoder made it. *Then* load it
and check. Getting this from the metadata rather than from a traceback is the habit.

**E2 — get the picture back.** The cache contains no pixels, but it does contain
`source_idx`. Take row 0 of the EuroSAT test cache, read its `source_idx`, and pull the
matching image out of `tanganke/eurosat` to display it. Two traps: the cache is shuffled,
so `source_idx` is not the row number; and the source dataset can change under you, so
pass the pinned `revision` from the cache's dataset card. Check the label you recover
matches the label in the cache.

**E3 — predict, then measure.** Before running anything: will `resnet-18` beat
`dinov2-small` on DTD (textures) by more or less than it does on flowers? Write down a
number, then use `two_line_head` to check it. You will meet the answer again in
notebook 04.